# 🧠 Logic Fine-Tuning: Qwen3-8B × Unsloth
**Pipeline:** Data Unpacking → LoRA Training → Chain-of-Thought Alignment → Evaluation

> Dataset: `Logic_Based_Educational_Queries-2.json` | 411 indices | FOL + NL premises  
> Target: **≥60% accuracy** trên toàn bộ test set

---
## 📋 Fixes so với phiên bản cũ
| # | Vấn đề cũ | Fix mới |
|---|-----------|---------|
| 1 | Qwen2.5-7B | **Qwen3-8B** (mạnh hơn về reasoning) |
| 2 | `formatting_prompts_func` định nghĩa 2 lần | Chỉ 1 lần, đúng signature |
| 3 | `train_on_responses_only` dùng sai token markers | Dùng đúng `<|im_start|>` cho Qwen3 |
| 4 | `dataset_text_field="text"` nhưng `formatting_func` trả list | Fix để SFTTrainer nhận đúng |
| 5 | LR=1e-5 quá thấp (under-fit) | **LR=2e-4** (chuẩn LoRA) |
| 6 | Evaluation dùng raw_data[-3:] không đúng split | Fix dùng đúng test split |
| 7 | `extract_final_answer` regex yếu | Regex mạnh hơn + fallback |
| 8 | Inference thiếu `repetition_penalty` | Thêm penalty để tránh loop |
| 9 | Evaluate chỉ trên vài sample | **Evaluate toàn bộ 411 indices** |

> Kaggle-ready edit: dataset/model paths now point to chồng's `test-pipeline` and local Qwen3-8B input, with fallback resolver for Kaggle mount quirks.

In [ ]:
%%capture
# Unsloth với CUDA 12.1 (T4 / A100 Kaggle)
!pip install unsloth
!pip install --upgrade --no-cache-dir trl peft accelerate bitsandbytes
!pip install datasets transformers sentencepiece protobuf

In [ ]:
import json
import os
import re
import random
import copy
from collections import Counter
from typing import List, Dict, Any, Optional
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# Seed cố định để reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
#  GLOBAL CONFIG — MATCH v14 ORIGINAL FOR FAIR COMPARISON
# ============================================================

def resolve_kaggle_path(*candidates: str) -> str:
    for p in candidates:
        if p and os.path.exists(p):
            return p
    print("⚠️ None of these paths exists yet:")
    for p in candidates:
        print("   -", p)
    return candidates[0]

DATA_PATH = resolve_kaggle_path(
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
)
EXTERNAL_TEST_PATH = resolve_kaggle_path(
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
)
MODEL_NAME = resolve_kaggle_path(
    "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1",
    "/kaggle/input/qwen-3/transformers/8b/1",
)
OUTPUT_DIR = "/kaggle/working/lora_output_qwen3_v14match"
os.makedirs(OUTPUT_DIR, exist_ok=True)
MAX_SEQ_LEN = 2048

# --- LoRA Config — MATCH v14 ---
LORA_R       = 16          # was 64 (overcapacity)
LORA_ALPHA   = 16          # was 128 (overcapacity)
LORA_DROPOUT = 0.0         # was 0.05 — v14 dùng 0
LORA_TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# --- Training Hyperparams — MATCH v14 ---
LEARNING_RATE    = 1e-4
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 4       # was 8 (effective 8, matches v14)
NUM_EPOCHS       = 2       # was 4 (overfit risk)
WARMUP_RATIO     = 0.05
LR_SCHEDULER     = "cosine"
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 1.0

# --- Data Split — record-level to avoid leak ---
TEST_RATIO  = 0.10
VAL_RATIO   = 0.15
TRAIN_RATIO = 0.75

# --- Adversarial Weighting — DISABLED to match v14 ---
UNKNOWN_OVERSAMPLE_FACTOR = 1   # was 3 — tắt để so sánh fair

print("✅ Config (v14-match):")
print(f"   DATA_PATH        : {DATA_PATH}")
print(f"   EXTERNAL_TEST    : {EXTERNAL_TEST_PATH}")
print(f"   Model            : {MODEL_NAME}")
print(f"   LoRA r/alpha     : {LORA_R}/{LORA_ALPHA}  (was 64/128)")
print(f"   LoRA dropout     : {LORA_DROPOUT}  (was 0.05)")
print(f"   Effective BS     : {BATCH_SIZE * GRAD_ACCUM_STEPS}  (was 16)")
print(f"   Epochs           : {NUM_EPOCHS}  (was 4)")
print(f"   Oversample       : {UNKNOWN_OVERSAMPLE_FACTOR}x  (was 3x — DISABLED)")
print(f"   Split            : train/val/test = {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")


In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data: List[Dict] = json.load(f)

print(f"📦 Tổng số indices: {len(raw_data)}")
print("\n--- Cấu trúc 1 index mẫu ---")
sample = raw_data[0]
print(f"  idx            : {sample['idx']}")
print(f"  premises-FOL   : {len(sample['premises-FOL'])} premises")
print(f"  premises-NL    : {len(sample['premises-NL'])} premises")
print(f"  questions      : {len(sample['questions'])} questions")
print(f"  answers        : {sample['answers']}")

all_answers = []
for item in raw_data:
    all_answers.extend(item["answers"])

answer_dist = Counter(all_answers)
total_ans = sum(answer_dist.values())
print(f"\n📊 Phân phối nhãn (tổng {total_ans} mẫu sau unpack):")
for label, count in sorted(answer_dist.items()):
    pct = count / total_ans * 100
    bar = "█" * int(pct / 2)
    print(f"  {label:10s}: {count:4d} ({pct:5.1f}%) {bar}")

In [ ]:
SYSTEM_PROMPT = """You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).
Given a set of premises (in natural language and FOL notation), you must:
1. Analyze the logical structure of each premise carefully.
2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation, existential instantiation, hypothetical syllogism.
3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.
4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.
5. For Yes/No questions: verify the statement logically.
6. If the premises are INSUFFICIENT to determine a unique conclusion, your Final Answer MUST be exactly: Unknown
7. Never hallucinate a conclusion that is not entailed by the premises.
Format your response EXACTLY as:
### Step-by-Step Reasoning
<your reasoning here>
### Final Answer
<A or B or C or D or Yes or No or Unknown>"""


def format_premises(fol_list: List[str], nl_list: List[str]) -> str:
    lines = ["### Premises"]
    for i, (fol, nl) in enumerate(zip(fol_list, nl_list), 1):
        lines.append(f"P{i}. [NL]  {nl}")
        lines.append(f"   [FOL] {fol}")
    return "\n".join(lines)


def build_user_message(fol_list: List[str], nl_list: List[str], question: str) -> str:
    premises_block = format_premises(fol_list, nl_list)
    return f"{premises_block}\n\n### Question\n{question}"


def build_assistant_message(explanation: str, answer: str) -> str:
    """Answer-Last: CoT trước, đáp án sau để model học suy luận."""
    return (
        f"### Step-by-Step Reasoning\n{explanation}\n\n"
        f"### Final Answer\n{answer}"
    )


print("✅ Prompt templates defined.")
s = raw_data[0]
print("\n--- User Message preview ---")
print(build_user_message(s["premises-FOL"], s["premises-NL"], s["questions"][0])[:400])
print("\n--- Assistant Message preview ---")
print(build_assistant_message(s["explanation"][0], s["answers"][0])[:300])

In [ ]:
def unpack_dataset(raw_data: List[Dict]) -> List[Dict]:
    """Mỗi (index × question) → 1 training sample độc lập."""
    samples = []
    for entry in raw_data:
        fol_list     = entry["premises-FOL"]
        nl_list      = entry["premises-NL"]
        questions    = entry["questions"]
        answers      = entry["answers"]
        explanations = entry["explanation"]

        assert len(questions) == len(answers) == len(explanations), (
            f"Mismatch in entry idx={entry.get('idx')}: "
            f"q={len(questions)}, a={len(answers)}, exp={len(explanations)}"
        )

        for q, a, exp in zip(questions, answers, explanations):
            user_msg      = build_user_message(fol_list, nl_list, q)
            assistant_msg = build_assistant_message(exp, a)
            sample = {
                "messages": [
                    {"role": "system",    "content": SYSTEM_PROMPT},
                    {"role": "user",      "content": user_msg},
                    {"role": "assistant", "content": assistant_msg},
                ],
                "answer_label": a,
                "is_unknown":   a.strip().lower() == "unknown",
                # Lưu gốc để evaluate toàn bộ 411 indices sau
                "_raw_entry_idx": id(entry),
            }
            samples.append(sample)
    return samples


flat_samples = unpack_dataset(raw_data)
unknown_count = sum(1 for s in flat_samples if s["is_unknown"])
print(f"📦 Tổng samples sau unpack  : {len(flat_samples)}")
print(f"   Nhãn Unknown             : {unknown_count} ({unknown_count/len(flat_samples)*100:.1f}%)")
print(f"   Nhãn có đáp án           : {len(flat_samples) - unknown_count}")

In [ ]:
def apply_adversarial_oversampling(
    samples: List[Dict],
    factor: int = UNKNOWN_OVERSAMPLE_FACTOR,
    seed: int = SEED
) -> List[Dict]:
    rng = random.Random(seed)
    unknown_samples = [s for s in samples if s["is_unknown"]]
    known_samples   = [s for s in samples if not s["is_unknown"]]

    augmented_unknowns = unknown_samples * factor
    combined = known_samples + augmented_unknowns
    rng.shuffle(combined)

    new_unknown_count = sum(1 for s in combined if s["is_unknown"])
    print(f"✅ Sau Adversarial Oversampling (factor={factor}x):")
    print(f"   Tổng samples  : {len(combined)}")
    print(f"   Unknown       : {new_unknown_count} ({new_unknown_count/len(combined)*100:.1f}%)")
    print(f"   Known answers : {len(combined) - new_unknown_count}")
    return combined


augmented_samples = apply_adversarial_oversampling(flat_samples)

In [ ]:
# ============================================================
#  RECORD-LEVEL SPLIT — phải tách test set TRƯỚC khi oversample
#  để tránh leak vào eval (chính là bug của notebook 92.4% gốc)
# ============================================================

def split_records(raw_data, test_ratio=TEST_RATIO, val_ratio=VAL_RATIO, seed=SEED):
    rng = random.Random(seed)
    indices = list(range(len(raw_data)))
    rng.shuffle(indices)
    n = len(indices)
    n_test = int(n * test_ratio)
    n_val  = int(n * val_ratio)
    test_idx  = indices[:n_test]
    val_idx   = indices[n_test : n_test + n_val]
    train_idx = indices[n_test + n_val :]
    return ([raw_data[i] for i in train_idx],
            [raw_data[i] for i in val_idx],
            [raw_data[i] for i in test_idx])

train_records, val_records, test_records = split_records(raw_data)

print(f"📊 Record-level split (records | questions):")
print(f"   train : {len(train_records):3d} records  | {sum(len(r['questions']) for r in train_records):4d} questions")
print(f"   val   : {len(val_records):3d} records  | {sum(len(r['questions']) for r in val_records):4d} questions")
print(f"   test  : {len(test_records):3d} records  | {sum(len(r['questions']) for r in test_records):4d} questions  ← HELD-OUT, never seen")

# Unpack each split into flat samples
flat_train = unpack_dataset(train_records)
flat_val   = unpack_dataset(val_records)
flat_test  = unpack_dataset(test_records)

# Oversample ONLY on train (val/test must reflect true distribution)
flat_train_aug = apply_adversarial_oversampling(flat_train, factor=UNKNOWN_OVERSAMPLE_FACTOR)

# Build HuggingFace DatasetDict for SFTTrainer
def to_hf(lst):
    return Dataset.from_list([{"messages": s["messages"]} for s in lst])

dataset = DatasetDict({
    "train":      to_hf(flat_train_aug),
    "validation": to_hf(flat_val),
})

print(f"\n📦 Final SFT splits (after oversample={UNKNOWN_OVERSAMPLE_FACTOR}x):")
print(f"   train      : {len(dataset['train']):4d} samples")
print(f"   validation : {len(dataset['validation']):4d} samples")
print(f"   test_records: held out for final eval (NOT in dataset)")

# Track identity của từng record → dùng để tag split trong eval
test_entry_ids  = {id(r) for r in test_records}
val_entry_ids   = {id(r) for r in val_records}
train_entry_ids = {id(r) for r in train_records}
print(f"   ID sets: train={len(train_entry_ids)} val={len(val_entry_ids)} test={len(test_entry_ids)}")


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

# FIX #3: Qwen3 dùng chat template "qwen-2.5" (tương thích Qwen3)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Vocab size    : {tokenizer.vocab_size}")
print(f"   Max seq length: {MAX_SEQ_LEN}")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = LORA_TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    use_rslora                 = False,
    loftq_config               = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied:")
print(f"   Trainable params: {trainable:,} ({trainable/total_p*100:.2f}%)")
print(f"   Total params    : {total_p:,}")
print(f"   Target modules  : {LORA_TARGET_MODULES}")

In [ ]:
# FIX #2 & #4: Chỉ định nghĩa 1 lần, đúng signature cho SFTTrainer
# SFTConfig với dataset_text_field="text" yêu cầu formatting_func trả về dict
# Nhưng khi dùng formatting_func KHÔNG có dataset_text_field → trả list of str

def formatting_prompts_func(examples):
    """
    Trả về list of strings (không dict).
    SFTTrainer với formatting_func sẽ tự tạo cột 'text' từ list này.
    """
    convos = examples["messages"]
    # Unsloth có thể gửi single sample (list of dicts) thay vì batch
    if convos and isinstance(convos[0], dict):
        convos = [convos]

    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize              = False,
            add_generation_prompt = False,
        )
        for convo in convos
    ]
    return texts   # list of str, không phải dict


# Verify formatting
sample_text = formatting_prompts_func({"messages": dataset["train"][0]["messages"]})
print("--- Formatted sample (đầu 600 ký tự) ---")
if isinstance(sample_text, list):
    print(sample_text[0][:600])
else:
    print(sample_text[:600])
print("\n[... truncated ...]")

In [ ]:
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm
import torch

# ── Custom Progress Callback ───────────────────────────────────────
class TqdmProgressCallback(TrainerCallback):
    def __init__(self):
        self.pbar = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="🚀 Training",
                         unit="step", colour="green")
    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.update(1)
            if state.log_history:
                self.pbar.set_postfix({"loss": f"{state.log_history[-1].get('loss', 0):.4f}"})
    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()

# ── Áp dụng Chat Template trước khi train để tránh lỗi Jinja2 ──
def apply_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return example

train_ds = dataset["train"].map(apply_template)
eval_ds  = dataset["validation"].map(apply_template)

_steps_per_epoch = max(1, len(train_ds) // (BATCH_SIZE * GRAD_ACCUM_STEPS))
_logging_steps   = max(1, min(10, _steps_per_epoch // 10))
# FIX #5: Không dùng dataset_text_field khi đã dùng formatting_func
# FIX #6: Thêm các tham số ổn định training
training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    # dataset_text_field KHÔNG set khi dùng formatting_func
    packing                     = False,
    padding_free                = False,
    assistant_only_loss         = False,  # Tắt để tránh xung đột với train_on_responses_only
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    num_train_epochs            = NUM_EPOCHS,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = MAX_GRAD_NORM,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 20,
    eval_strategy               = "steps",
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    seed                        = SEED,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    optim                       = "adamw_8bit",
    push_to_hub                 = False,
)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset["train"],
    eval_dataset     = dataset["validation"],
    formatting_func  = formatting_prompts_func,
    args             = training_args,
)

# FIX #3: Qwen3 tokens đúng cho train_on_responses_only
# Verify token markers trước khi dùng
sample_formatted = tokenizer.apply_chat_template(
    dataset["train"][0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print("Token markers check:")
print("  <|im_start|>user    present:", "<|im_start|>user" in sample_formatted)
print("  <|im_start|>assistant present:", "<|im_start|>assistant" in sample_formatted)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

print("\n✅ SFTTrainer initialized successfully!")
print(f"   Train samples : {len(dataset['train'])}")
print(f"   Val samples   : {len(dataset['validation'])}")

In [ ]:
print("🚀 Starting fine-tuning...")
print("=" * 60)

trainer_stats = trainer.train()

print("\n" + "=" * 60)
print("✅ Training complete!")
print(f"   Total steps    : {trainer_stats.global_step}")
print(f"   Training time  : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Samples/second : {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"   Final loss     : {trainer_stats.metrics['train_loss']:.4f}")

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"   Peak VRAM used : {peak:.2f} GB")

In [ ]:
LORA_SAVE_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
os.makedirs(LORA_SAVE_PATH, exist_ok=True)

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

saved_files = os.listdir(LORA_SAVE_PATH)
total_size_mb = sum(
    os.path.getsize(os.path.join(LORA_SAVE_PATH, f))
    for f in saved_files
) / 1e6

print("✅ LoRA adapter saved.")
print(f"   Path  : {LORA_SAVE_PATH}")
print(f"   Files : {saved_files}")
print(f"   Size  : {total_size_mb:.1f} MB")

In [ ]:
# Chuyển sang inference mode
FastLanguageModel.for_inference(model)


def predict(fol_list: List[str], nl_list: List[str], question: str,
            max_new_tokens: int = 512, temperature: float = 0.0) -> str:
    """Inference với repetition_penalty để tránh loop (FIX #8)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_message(fol_list, nl_list, question)},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens      = max_new_tokens,
            temperature         = temperature,
            do_sample           = temperature > 0,
            repetition_penalty  = 1.0,    # FIX #8: tránh vòng lặp token
            pad_token_id        = tokenizer.eos_token_id,
            eos_token_id        = tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("✅ predict() function ready with repetition_penalty=1.1")

In [ ]:
def extract_final_answer(model_output: str) -> str:
    """
    FIX #7: Regex mạnh hơn — nhiều pattern fallback để không miss.
    Priority order: Final Answer block > inline answer patterns.
    """
    text = model_output.strip()

    # Pattern 1: phần sau '### Final Answer'
    match = re.search(
        r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)",
        text, re.IGNORECASE
    )
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        if re.match(r"^unknown", ans, re.IGNORECASE):
            return "Unknown"
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        # Yes/No
        if re.match(r"^yes", ans, re.IGNORECASE):
            return "Yes"
        if re.match(r"^no", ans, re.IGNORECASE):
            return "No"

    # Pattern 2: "Answer: X" hoặc "answer is X"
    match2 = re.search(
        r"(?:answer\s*(?:is|:)\s*)([A-D]|unknown|yes|no)",
        text, re.IGNORECASE
    )
    if match2:
        g = match2.group(1).strip()
        if g.lower() == "unknown": return "Unknown"
        if g.lower() == "yes":    return "Yes"
        if g.lower() == "no":     return "No"
        return g.upper()

    # Pattern 3: standalone letter near end of text
    last_500 = text[-500:]
    match3 = re.findall(r"\b([A-D])\b", last_500)
    if match3:
        return match3[-1].upper()

    # Pattern 4: Unknown/Yes/No anywhere
    if re.search(r"\bunknown\b", text, re.IGNORECASE):
        return "Unknown"
    if re.search(r"\byes\b", text, re.IGNORECASE):
        return "Yes"
    if re.search(r"\bno\b", text, re.IGNORECASE):
        return "No"

    return "UNPARSEABLE"


# Test
test_cases = [
    ("### Final Answer\nA", "A"),
    ("### Final Answer\nUnknown", "Unknown"),
    ("### Final Answer\nYes", "Yes"),
    ("some reasoning\n### Final Answer\nB. Because...", "B"),
    ("the answer is C", "C"),
    ("UNPARSEABLE output xyz", "UNPARSEABLE"),
]
print("Testing extract_final_answer:")
for text, expected in test_cases:
    got = extract_final_answer(text)
    status = "✅" if got == expected else "❌"
    print(f"  {status} '{text[:40]}'  →  got={got}, expected={expected}")

In [ ]:
# ============================================================
#  EVAL: FULL raw_data + HELD-OUT TEST + SAMPLE PRINT
# ============================================================
import random as _rng

print("📊 Phase 1: Eval trên TOÀN BỘ raw_data (630 câu — bao gồm train)...")
print("=" * 65)

all_results = []

for entry_i, entry in enumerate(raw_data):
    # Tag split
    eid = id(entry)
    split = "test" if eid in test_entry_ids else ("val" if eid in val_entry_ids else "train")

    for q_i, (q, gold_ans, gold_exp) in enumerate(
        zip(entry["questions"], entry["answers"], entry["explanation"])
    ):
        output = predict(
            entry["premises-FOL"],
            entry["premises-NL"],
            q,
            max_new_tokens=256,
            temperature=0.0,
        )
        pred = extract_final_answer(output)
        is_correct = (pred.strip().lower() == gold_ans.strip().lower())

        all_results.append({
            "entry_i": entry_i, "q_i": q_i, "split": split,
            "question": q, "gold": gold_ans, "pred": pred,
            "correct": is_correct,
            "gold_exp": gold_exp[:300],
            "premises_nl": entry["premises-NL"],
            "premises_fol": entry["premises-FOL"],
        })

    if (entry_i + 1) % 50 == 0:
        done = [r for r in all_results if r["entry_i"] <= entry_i]
        print(f"  [{entry_i+1:3d}/318]  running: {100*sum(r['correct'] for r in done)/len(done):.1f}%")

# ── Split results ──────────────────────────────────────────────────────────
train_results = [r for r in all_results if r["split"] == "train"]
val_results   = [r for r in all_results if r["split"] == "val"]
test_results  = [r for r in all_results if r["split"] == "test"]

def _acc(lst):
    return 100 * sum(r["correct"] for r in lst) / len(lst) if lst else 0.0

full_acc  = _acc(all_results)
train_acc = _acc(train_results)
val_acc   = _acc(val_results)
test_acc  = _acc(test_results)

# ── Result table ──────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  ACCURACY COMPARISON (số thật để phát hiện overfit)")
print("=" * 65)
print(f"  FULL data   (train+val+test): {sum(r['correct'] for r in all_results):4d}/{len(all_results):4d} = {full_acc:.1f}%")
print(f"  TRAIN only  (seen in train) : {sum(r['correct'] for r in train_results):4d}/{len(train_results):4d} = {train_acc:.1f}%")
print(f"  VAL only                    : {sum(r['correct'] for r in val_results):4d}/{len(val_results):4d} = {val_acc:.1f}%")
print(f"  TEST only   (NEVER seen)    : {sum(r['correct'] for r in test_results):4d}/{len(test_results):4d} = {test_acc:.1f}%  ← số thật")
print(f"\n  GAP (train - test)          : {train_acc - test_acc:+.1f}pp")
if train_acc - test_acc > 15:
    print(f"  ← OVERFIT NẶNG: model đang nhớ train, không generalize")
elif train_acc - test_acc > 7:
    print(f"  ← Overfit vừa: gap lớn hơn chấp nhận được")
elif train_acc - test_acc > 3:
    print(f"  ← Overfit nhẹ: bình thường với dataset nhỏ")
else:
    print(f"  ← OK: model generalize tốt")

# ── Per-class breakdown ────────────────────────────────────────────────────
from collections import Counter, defaultdict
print("\n  Per-class (TRAIN vs TEST):")
print(f"  {'Label':>10} {'Train acc':>10} {'Test acc':>10} {'Count (test)':>14}")
print(f"  {'-'*48}")
all_labels = sorted(set(r["gold"] for r in all_results))
for label in all_labels:
    tr = [r for r in train_results if r["gold"] == label]
    te = [r for r in test_results  if r["gold"] == label]
    ta = f"{_acc(tr):.0f}%" if tr else "n/a"
    ea = f"{_acc(te):.0f}%" if te else "n/a"
    flag = " ← Z3 underfit" if label == "No" and te and _acc(te) < 60 else ""
    print(f"  {label:>10} {ta:>10} {ea:>10} {len(te):>14}{flag}")

# ── Print sample examples ──────────────────────────────────────────────────
_seed = _rng.Random(42)

def print_examples(title, sample_list, n=5):
    print(f"\n{'='*65}")
    print(f"  {title}")
    print(f"{'='*65}")
    chosen = _seed.sample(sample_list, min(n, len(sample_list)))
    for i, r in enumerate(chosen, 1):
        tick = "✓" if r["correct"] else "✗"
        print(f"\n  [{i}] {tick} entry={r['entry_i']} q={r['q_i']}  Gold: {r['gold']:8s}  Pred: {r['pred']}")
        print(f"      Q: {r['question'][:100]}")
        print(f"      Gold explanation: {r['gold_exp'][:150]}...")
        if not r["correct"]:
            print(f"      ⚠ WRONG → Expected '{r['gold']}' but got '{r['pred']}'")

print_examples("5 câu mẫu từ TRAIN (đã học)", train_results, n=5)
print_examples("5 câu mẫu từ TEST (chưa thấy)", test_results, n=5)

# Wrong samples breakdown
wrong_test = [r for r in test_results if not r["correct"]]
print(f"\n  Lỗi trên test: {len(wrong_test)}/{len(test_results)}")
if wrong_test:
    confusion_test = defaultdict(Counter)
    for r in test_results:
        confusion_test[r["gold"]][r["pred"]] += 1
    print("  Confusion (test set):")
    for gold_label in sorted(confusion_test):
        row = confusion_test[gold_label]
        print(f"    {gold_label:>10}: ", end="")
        for pred_label, cnt in sorted(row.items()):
            print(f"{pred_label}={cnt} ", end="")
        print()

# Save full results
import json as _json
_json.dump({
    "summary": {
        "full_acc": full_acc, "train_acc": train_acc,
        "val_acc": val_acc, "test_acc": test_acc,
        "gap_train_test": train_acc - test_acc,
        "n_train": len(train_results), "n_val": len(val_results),
        "n_test":  len(test_results),
    },
    "all_results": all_results,
}, open("/kaggle/working/eval_v14match_full.json", "w", encoding="utf-8"),
   ensure_ascii=False, indent=2)
print(f"\n📁 Saved: /kaggle/working/eval_v14match_full.json")


In [ ]:
# ============================================================
#  OPTIONAL: Eval on EXTERNAL test set (OOD — generated_v4style_300)
#  Đây là test set thực sự ngoài training distribution
# ============================================================

RUN_EXTERNAL_TEST = True   # đổi False nếu không muốn chạy

if RUN_EXTERNAL_TEST and os.path.exists(EXTERNAL_TEST_PATH):
    print(f"📊 Evaluating on EXTERNAL test set: {EXTERNAL_TEST_PATH}")
    print("=" * 60)
    with open(EXTERNAL_TEST_PATH, "r", encoding="utf-8") as f:
        ext_data = json.load(f)
    print(f"  External records: {len(ext_data)}  questions: {sum(len(r['questions']) for r in ext_data)}")

    ext_correct, ext_total = 0, 0
    ext_log = []
    for entry_i, entry in enumerate(ext_data):
        for q_i, (q, expected_ans, _) in enumerate(
            zip(entry["questions"], entry["answers"], entry["explanation"])
        ):
            output = predict(entry["premises-FOL"], entry["premises-NL"], q,
                             max_new_tokens=256, temperature=0.0)
            pred = extract_final_answer(output)
            is_correct = (pred.strip().lower() == expected_ans.strip().lower())
            ext_total += 1
            if is_correct: ext_correct += 1
            ext_log.append({"entry_i": entry_i, "q_i": q_i,
                            "expected": expected_ans, "pred": pred, "correct": is_correct})
        if (entry_i + 1) % 50 == 0:
            print(f"  [{entry_i+1:3d}/{len(ext_data)}] running: {100*ext_correct/ext_total:.1f}%")

    ext_acc = 100 * ext_correct / ext_total
    print(f"\n✅ EXTERNAL TEST: {ext_correct}/{ext_total} = {ext_acc:.1f}%")
    json.dump({"metrics": {"acc": ext_acc, "n": ext_total},
               "results": ext_log},
              open("/kaggle/working/eval_v14match_external.json","w",encoding="utf-8"),
              ensure_ascii=False, indent=2)
    print(f"📁 Saved: /kaggle/working/eval_v14match_external.json")

    # Compare gap
    if 'overall_acc' in dir():
        gap = overall_acc - ext_acc
        print(f"\n📊 GAP analysis:")
        print(f"   Held-out test : {overall_acc:.1f}%")
        print(f"   External test : {ext_acc:.1f}%")
        print(f"   Gap           : {gap:+.1f}pp  ← nếu > 10pp = overfit pattern in-distribution")
else:
    print("(External test skipped)")


In [ ]:
# Phân tích lỗi để hiểu pattern
wrong_samples = [r for r in results_log if not r["correct"]]
print(f"\n🔍 Error Analysis ({len(wrong_samples)} wrong predictions)")
print("=" * 50)

# Confusion matrix
from collections import defaultdict
confusion = defaultdict(Counter)
for r in results_log:
    confusion[r["expected"]][r["pred"]] += 1

print("\nConfusion (expected → predicted):")
for expected_label in sorted(confusion.keys()):
    row = confusion[expected_label]
    total_row = sum(row.values())
    print(f"  {expected_label:8s} (n={total_row:3d}): ", end="")
    for pred_label, count in sorted(row.items()):
        print(f"{pred_label}={count}", end="  ")
    print()

# Mẫu sai đầu tiên
print(f"\n--- 3 mẫu sai đầu tiên ---")
for r in wrong_samples[:3]:
    entry = raw_data[r["idx"]]
    q     = entry["questions"][r["q_i"]]
    print(f"  idx={r['idx']}, q={r['q_i']}")
    print(f"  Q: {q[:80]}...")
    print(f"  Expected: {r['expected']} | Predicted: {r['pred']}")
    print()

In [ ]:
# Optional: Push lên HuggingFace Hub
PUSH_TO_HUB = False
HF_REPO_ID  = "Barakuga/qwen3-8b-logic-lora"

if PUSH_TO_HUB:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")

    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        print(f"✅ Pushed to: https://huggingface.co/{HF_REPO_ID}")
    else:
        print("⚠️  HF_TOKEN not found. Set via Kaggle Secrets.")
else:
    print("ℹ️  PUSH_TO_HUB=False. Đặt True để push lên HuggingFace Hub.")

## Z3 add-on: conservative formal gate
Run these cells after the LoRA model is trained and `predict()` / `extract_final_answer()` are available.


In [ ]:
# ============================================================
#  Z3 ADD-ON v1 — Conservative precision gate for exact_70.ipynb
#  Place after cell 16 (extract_final_answer) or after cell 17.
# ============================================================

# If z3-solver is not installed on Kaggle:
try:
    import z3
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'])
    import z3

import re, json, os
from collections import Counter, defaultdict
from typing import List, Dict, Any, Tuple, Optional

Z3_TIMEOUT_MS = 1500
Z3_ENABLE_OVERRIDE = True
Z3_OVERRIDE_MODE = 'qwen_first'  # qwen_first: override only when Z3 is definite and trusted

# ---------- FOL tokenizer / parser ----------

def _norm_fol(s: str) -> str:
    s = str(s).strip()
    s = s.replace('∀', ' forall ').replace('∃', ' exists ')
    s = s.replace('¬', ' ~ ').replace('!', ' ~ ')
    s = s.replace('→', ' -> ').replace('⇒', ' -> ')
    s = s.replace('↔', ' <-> ')
    s = s.replace('∧', ' & ').replace('∨', ' | ')
    # normalize quantifier glued forms: forallx -> forall x rarely appears
    s = re.sub(r'\bFor all\b', 'forall', s, flags=re.I)
    s = re.sub(r'\bThere exists\b', 'exists', s, flags=re.I)
    return s

_TOKEN_RE = re.compile(r"forall|exists|<->|->|[A-Za-z_][A-Za-z0-9_]*|[(),.~&|]", re.I)

def _tok(s: str) -> List[str]:
    return _TOKEN_RE.findall(_norm_fol(s))

class Z3FOLParser:
    def __init__(self):
        self.predicates = {}   # (name, arity) -> z3 Function
        self.consts = {}       # name -> z3 Int const
        self.sort = z3.IntSort()

    def _pred(self, name, arity):
        key = (name, arity)
        if key not in self.predicates:
            self.predicates[key] = z3.Function(name, *([self.sort] * arity), z3.BoolSort())
        return self.predicates[key]

    def _const(self, name):
        if name not in self.consts:
            self.consts[name] = z3.Int(name)
        return self.consts[name]

    def parse(self, text: str):
        self.tokens = _tok(text)
        self.i = 0
        if not self.tokens:
            raise ValueError('empty formula')
        out = self._parse_iff({})
        if self.i != len(self.tokens):
            raise ValueError(f'trailing tokens: {self.tokens[self.i:]}')
        return out

    def _peek(self):
        return self.tokens[self.i] if self.i < len(self.tokens) else None

    def _eat(self, t=None):
        cur = self._peek()
        if t is not None and cur != t:
            raise ValueError(f'expected {t}, got {cur}')
        self.i += 1
        return cur

    def _parse_iff(self, env):
        left = self._parse_imp(env)
        while self._peek() == '<->':
            self._eat('<->')
            right = self._parse_imp(env)
            left = left == right
        return left

    def _parse_imp(self, env):
        left = self._parse_or(env)
        if self._peek() == '->':
            self._eat('->')
            right = self._parse_imp(env)  # right associative
            return z3.Implies(left, right)
        return left

    def _parse_or(self, env):
        xs = [self._parse_and(env)]
        while self._peek() == '|':
            self._eat('|')
            xs.append(self._parse_and(env))
        return xs[0] if len(xs) == 1 else z3.Or(*xs)

    def _parse_and(self, env):
        xs = [self._parse_unary(env)]
        while self._peek() == '&':
            self._eat('&')
            xs.append(self._parse_unary(env))
        return xs[0] if len(xs) == 1 else z3.And(*xs)

    def _parse_unary(self, env):
        cur = self._peek()
        if cur in ('~', '.'):
            self._eat(cur)
            return z3.Not(self._parse_unary(env))
        if cur in ('forall', 'exists'):
            q = self._eat().lower()
            var = self._eat()
            zvar = z3.Int(var)
            new_env = dict(env); new_env[var] = zvar
            # optional punctuation after quantifier variable
            if self._peek() in ('.', ','):
                self._eat()
            body = self._parse_unary(new_env)
            return z3.ForAll([zvar], body) if q == 'forall' else z3.Exists([zvar], body)
        if cur == '(':
            self._eat('(')
            f = self._parse_iff(env)
            self._eat(')')
            return f
        return self._parse_atom(env)

    def _parse_term(self, env):
        name = self._eat()
        if name in env:
            return env[name]
        # single-letter lowercase x/y/z often variables if not bound; keep as free Int const but flag later
        return self._const(name)

    def _parse_atom(self, env):
        name = self._eat()
        if not re.match(r'^[A-Za-z_]', name):
            raise ValueError(f'bad atom name {name}')
        args = []
        if self._peek() == '(':
            self._eat('(')
            if self._peek() != ')':
                while True:
                    args.append(self._parse_term(env))
                    if self._peek() == ',':
                        self._eat(',')
                        continue
                    break
            self._eat(')')
        else:
            # propositional atom
            pass
        return self._pred(name, len(args))(*args)


def parse_premises_to_z3(fol_list: List[str]) -> Tuple[Optional[Z3FOLParser], List[Any], List[str]]:
    parser = Z3FOLParser()
    axioms, bad = [], []
    for fol in fol_list:
        try:
            axioms.append(parser.parse(fol))
        except Exception as e:
            bad.append(f'{fol}  // {type(e).__name__}: {e}')
    if len(bad) > 0:
        # conservative: if any premise fails, do not trust Z3 for this item
        return None, [], bad
    return parser, axioms, bad


def z3_entails(axioms, stmt, timeout_ms=Z3_TIMEOUT_MS) -> str:
    # returns Yes if premises entail stmt; No if premises entail not stmt; Unknown otherwise
    s = z3.Solver(); s.set(timeout=timeout_ms)
    s.add(*axioms); s.add(z3.Not(stmt))
    r1 = s.check()
    if r1 == z3.unsat:
        return 'Yes'
    s = z3.Solver(); s.set(timeout=timeout_ms)
    s.add(*axioms); s.add(stmt)
    r2 = s.check()
    if r2 == z3.unsat:
        return 'No'
    return 'Unknown'


def extract_statement_fol(question: str) -> Optional[str]:
    q = str(question)
    # common pattern: Statement: ∃x U(x) / Statement: '...'
    m = re.search(r"Statement\s*:\s*['\"]?(.+?)['\"]?\s*$", q, flags=re.I | re.S)
    cand = m.group(1).strip() if m else q.strip()
    # only trust if formula-looking
    if re.search(r'[∀∃¬→∧∨]|\b(forall|exists)\b|[A-Za-z_]\w*\s*\(', cand):
        # remove trailing English explanation/options if present
        cand = re.split(r'\n\s*[A-D][\).]', cand)[0].strip()
        return cand
    return None


def extract_mc_options_fol(question: str) -> Dict[str, str]:
    q = str(question)
    opts = {}
    # Capture A. ... until next B./C./D. or end
    for m in re.finditer(r'(?ms)^\s*([A-D])\s*[\).]\s*(.*?)(?=^\s*[A-D]\s*[\).]|\Z)', q):
        letter, text = m.group(1).upper(), m.group(2).strip()
        # Extract formula inside quotes if present, otherwise whole option if formula-looking
        qm = re.search(r"['\"]([^'\"]*[∀∃¬→∧∨][^'\"]*|[^'\"]*[A-Za-z_]\w*\s*\([^'\"]*)['\"]", text)
        cand = qm.group(1).strip() if qm else text
        if re.search(r'[∀∃¬→∧∨]|\b(forall|exists)\b|[A-Za-z_]\w*\s*\(', cand):
            opts[letter] = cand
    return opts


def z3_predict_one(fol_list: List[str], question: str) -> Dict[str, Any]:
    parser, axioms, bad = parse_premises_to_z3(fol_list)
    if parser is None:
        return {'answer': 'Unknown', 'definite': False, 'mode': 'premise_parse_fail', 'bad': bad, 'conf': 0.0}

    # Multiple choice: choose exactly one entailed option, if unique.
    opts = extract_mc_options_fol(question)
    if len(opts) >= 2:
        entailed = []
        details = {}
        for letter, formula in opts.items():
            try:
                stmt = parser.parse(formula)
                verdict = z3_entails(axioms, stmt)
                details[letter] = verdict
                if verdict == 'Yes':
                    entailed.append(letter)
            except Exception as e:
                details[letter] = f'parse_fail:{e}'
        if len(entailed) == 1:
            return {'answer': entailed[0], 'definite': True, 'mode': 'mc_unique_entailed', 'details': details, 'conf': 0.95}
        return {'answer': 'Unknown', 'definite': False, 'mode': 'mc_not_unique', 'details': details, 'conf': 0.0}

    # Yes/No/Unknown statement FOL
    stmt_text = extract_statement_fol(question)
    if stmt_text:
        try:
            stmt = parser.parse(stmt_text)
            ans = z3_entails(axioms, stmt)
            return {'answer': ans, 'definite': ans in ('Yes', 'No'), 'mode': 'statement_fol', 'stmt': stmt_text, 'conf': 0.9 if ans in ('Yes','No') else 0.4}
        except Exception as e:
            return {'answer': 'Unknown', 'definite': False, 'mode': 'stmt_parse_fail', 'stmt': stmt_text, 'err': str(e), 'conf': 0.0}

    return {'answer': 'Unknown', 'definite': False, 'mode': 'no_formula_in_question', 'conf': 0.0}


def combine_qwen_z3(qwen_pred: str, z3_info: Dict[str, Any]) -> str:
    # Conservative rule: never let Z3 Unknown override Qwen; only definite Z3 can override.
    if Z3_ENABLE_OVERRIDE and z3_info.get('definite') and z3_info.get('answer') in {'A','B','C','D','Yes','No'}:
        return z3_info['answer']
    return qwen_pred

print('✅ Z3 add-on loaded. Conservative override only when parse is definite.')


In [ ]:
# ============================================================
#  Hybrid Qwen + Z3 — eval on HELD-OUT test_records ONLY
# ============================================================
print("📊 Evaluating Qwen + Z3 conservative gate on HELD-OUT test...")
print("=" * 60)

hybrid_correct = 0
qwen_correct = 0
total_eval = 0
hybrid_log = []
route_counter = Counter()

for entry_i, entry in enumerate(test_records):
    for q_i, (q, expected_ans, _) in enumerate(zip(entry["questions"], entry["answers"], entry["explanation"])):
        output = predict(entry["premises-FOL"], entry["premises-NL"], q,
                         max_new_tokens=256, temperature=0.0)
        qwen_pred = extract_final_answer(output)
        z3_info = z3_predict_one(entry["premises-FOL"], q)
        hybrid_pred = combine_qwen_z3(qwen_pred, z3_info)

        q_ok = qwen_pred.strip().lower() == expected_ans.strip().lower()
        h_ok = hybrid_pred.strip().lower() == expected_ans.strip().lower()
        qwen_correct += int(q_ok)
        hybrid_correct += int(h_ok)
        total_eval += 1

        route = 'z3_override' if hybrid_pred != qwen_pred else 'qwen_keep'
        route_counter[route] += 1
        route_counter[f"z3_mode::{z3_info.get('mode')}"] += 1

        hybrid_log.append({"entry_i": entry_i, "q_i": q_i, "expected": expected_ans,
                           "qwen_pred": qwen_pred, "z3_pred": z3_info.get('answer'),
                           "z3_definite": z3_info.get('definite'),
                           "z3_mode": z3_info.get('mode'),
                           "hybrid_pred": hybrid_pred,
                           "qwen_correct": q_ok, "hybrid_correct": h_ok})

print("\n" + "=" * 60)
print("✅ HELD-OUT QWEN vs QWEN+Z3 RESULTS")
print("=" * 60)
qwen_acc = qwen_correct / total_eval * 100
hybrid_acc = hybrid_correct / total_eval * 100
print(f"Qwen accuracy   : {qwen_correct}/{total_eval} = {qwen_acc:.1f}%")
print(f"Hybrid accuracy : {hybrid_correct}/{total_eval} = {hybrid_acc:.1f}%")
print(f"Delta           : {hybrid_acc - qwen_acc:+.1f} pp")

fixed  = [r for r in hybrid_log if (not r["qwen_correct"]) and r["hybrid_correct"]]
broken = [r for r in hybrid_log if r["qwen_correct"] and (not r["hybrid_correct"])]
print(f"\nFixed by Z3  : {len(fixed)}")
print(f"Broken by Z3 : {len(broken)}")

json.dump({"qwen_acc": qwen_acc, "hybrid_acc": hybrid_acc,
           "fixed_by_z3": len(fixed), "broken_by_z3": len(broken),
           "routing": dict(route_counter),
           "log": hybrid_log},
          open("/kaggle/working/eval_v14match_hybrid.json","w",encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(f"\n📁 Saved: /kaggle/working/eval_v14match_hybrid.json")
